# 02. Data Analysis — EDA + 맵매칭 + 복합위반 + 시나리오 100건
**v4 수정**: 시나리오 60건 → 100건 확대 (통계적 신뢰도 확보)
- description에서 초과속도 제거, 난이도 3단계 (Easy/Medium/Hard)
- 각 레벨별 최소 10건 이상 확보, 경계값·가중·복합 시나리오 보강

**의존**: 01_data_collection.py 완료, speed_matcher.py 존재

In [9]:
import os, json, pathlib, glob
import pandas as pd
import numpy as np
from collections import Counter

from config import (load_korean_csv, save_json, get_severity_level,
                    DTG_DIR, SPEED_DIR, SCENARIO_DIR, RESULTS_DIR,
                    SEVERITY_TABLE, VIOLATION_MAPPING)

## 1. DTG 초단위 EDA

In [10]:
body_file = next((f for f in glob.glob(str(DTG_DIR/'*.csv'))
                  if 'body' in f.lower() or '바디' in f), None)

df_body = None
if body_file:
    df_body = load_korean_csv(body_file)
    speed_col = [c for c in df_body.columns if '속도' in c and 'KMH' in c.upper()]
    print(f'=== 초단위 데이터: {df_body.shape} ===')
    if speed_col:
        sc = speed_col[0]
        print(f'  속도 평균: {df_body[sc].mean():.1f}, 최대: {df_body[sc].max()}, 정지: {(df_body[sc]==0).mean():.1%}')
    accel_cols = [c for c in df_body.columns if '가속도' in c]
    for ac in accel_cols:
        print(f'  {ac}: [{df_body[ac].min():.1f}, {df_body[ac].max():.1f}]')

=== 초단위 데이터: (1100, 14) ===
  속도 평균: 12.0, 최대: 70, 정지: 47.8%
  가속도 Vx: [-4.9, 8.6]
  가속도 Vy: [-4.9, 4.9]


## 2. 맵매칭

In [11]:
from speed_matcher import SpeedLimitMatcher

sign_files = sorted(glob.glob(str(SPEED_DIR / '*표지*')))
hw_files = sorted(glob.glob(str(SPEED_DIR / '*고속*')))
matcher = SpeedLimitMatcher(
    sign_csv_path=sign_files[0] if sign_files else None,
    highway_csv_path=hw_files[0] if hw_files else None)

if df_body is not None:
    gps_x_col = next((c for c in df_body.columns if 'GPS' in c and 'X' in c), None)
    gps_y_col = next((c for c in df_body.columns if 'GPS' in c and 'Y' in c), None)
    speed_col_name = next((c for c in df_body.columns if '속도' in c and 'KMH' in c.upper()), None)
    if gps_x_col and gps_y_col and speed_col_name:
        match_df = matcher.match_dtg_dataframe(df_body, speed_col=speed_col_name,
                    gps_x_col=gps_x_col, gps_y_col=gps_y_col, vehicle_weight_tons=5.0)
        save_json(matcher.get_match_report(), RESULTS_DIR / 'evaluation' / 'match_report.json')

✅ 표지판 데이터: 5600건 (속도:주행제한속도, 좌표:위도/경도)
✅ 고속도로 데이터: 95건

=== 맵매칭 결과 ===
  총 레코드: 1100건
  매칭 소스: {'statutory': 1100}
  과속 건수: 26건 (2.4%)
  과속 심각도 분포:
    level_1: 26건

  매칭률: sign=0, highway=0, statutory=1100
✅ 저장: results\evaluation\match_report.json


## 2-1. 논문 배경 통계 검증
논문에서 인용하는 핵심 통계 수치의 원본 데이터 출처를 검증합니다.

In [12]:
# ★ 논문 배경 통계 검증 — '2024년 화물차 교통사고 사망자 594명' 출처 확인
# 출처: 교통사고통계_20260324.xlsx (경찰청 교통사고통계, 2026.03.24 기준)
# 원본 URL: https://koroad.or.kr (도로교통공단 교통사고분석시스템 TAAS)

from config import STAT_DIR
import glob

stat_files = sorted(glob.glob(str(STAT_DIR / '*.xlsx')) + glob.glob(str(STAT_DIR / '*.csv')))
if not stat_files:
    # 프로젝트 루트에서도 탐색
    stat_files = sorted(glob.glob('교통사고통계*.xlsx'))

for sf in stat_files:
    if '교통사고통계' in sf:
        try:
            df_stat = pd.read_excel(sf, engine='openpyxl')
            cargo_death = df_stat[
                (df_stat.iloc[:, 0] == '화물차') & 
                (df_stat.iloc[:, 1].str.contains('사망', na=False))
            ]
            if len(cargo_death) > 0:
                deaths = int(cargo_death.iloc[0, 2])
                total_death = int(df_stat[
                    (df_stat.iloc[:, 0] == '합계') & 
                    (df_stat.iloc[:, 1].str.contains('사망', na=False))
                ].iloc[0, 2])
                print(f'=== 논문 배경 통계 검증 ===')
                print(f'  출처: {pathlib.Path(sf).name}')
                print(f'  2024년 전체 교통사고 사망자: {total_death:,}명')
                print(f'  2024년 화물차 사고 사망자:   {deaths:,}명 ({deaths/total_death:.1%})')
                assert deaths == 594, f'❌ 594명 불일치: 실제={deaths}'
                print(f'  ✅ 논문 인용값(594명) 검증 완료')
                print(f'  → 논문 인용 시: "경찰청 교통사고통계(TAAS, 2024)"')
        except Exception as e:
            print(f'  ⚠️ 통계 검증 실패: {e}')
            print(f'  → 교통사고통계_20260324.xlsx를 data/04_ACCIDENT_STAT/에 배치하세요')


=== 논문 배경 통계 검증 ===
  출처: 교통사고통계_20260324.xlsx
  2024년 전체 교통사고 사망자: 2,521명
  2024년 화물차 사고 사망자:   594명 (23.6%)
  ✅ 논문 인용값(594명) 검증 완료
  → 논문 인용 시: "경찰청 교통사고통계(TAAS, 2024)"


c:\Users\pc\anaconda3\envs\llm\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 3. Trip 위험운전행동 EDA

In [13]:
trip_file = next((f for f in glob.glob(str(DTG_DIR/'*.csv')) if 'trip' in f.lower()), None)
df_trip = None
found_behavior_cols = {}

if trip_file:
    df_trip = load_korean_csv(trip_file)
    behavior_keywords = {'과속':'speeding','급가속':'sudden_accel','급출발':'sudden_start',
        '급감속':'sudden_decel','급정지':'sudden_stop','급좌회전':'sudden_left_turn',
        '급우회전':'sudden_right_turn','급유턴':'sudden_uturn','급앞지르기':'sudden_overtake',
        '급진로변경':'sudden_lane_change'}
    for col in df_trip.columns:
        for kr, en in behavior_keywords.items():
            if kr in col and '건수' in col:
                found_behavior_cols[en] = col; break
    print(f'Trip: {df_trip.shape}, 위험운전행동 컬럼 {len(found_behavior_cols)}개')

Trip: (100, 66), 위험운전행동 컬럼 10개


## 4. 복합 위반 빈도

In [14]:
if df_trip is not None and found_behavior_cols:
    df_trip['n_viol'] = df_trip.apply(
        lambda row: sum(1 for c in found_behavior_cols.values() if row.get(c,0)>0), axis=1)
    df_w = df_trip[df_trip['n_viol']>0]
    compound = df_w[df_w['n_viol']>=2]
    print(f'복합 위반: {len(compound)}/{len(df_w)} ({len(compound)/max(len(df_w),1):.1%})')
    save_json({'compound_rate': len(compound)/max(len(df_w),1),
               'distribution': df_w['n_viol'].value_counts().to_dict()},
              RESULTS_DIR / 'evaluation' / 'compound_analysis.json')

복합 위반: 95/98 (96.9%)
✅ 저장: results\evaluation\compound_analysis.json


## 5. ★ 시나리오 100건 생성 (v4 — 변별력 확보)
**핵심 수정**:
- description에서 "Xkm/h 초과" 제거
- Easy: 제한속도 명시 (계산만 하면 됨)
- Medium: 제한속도 미명시, 화물차 특수규정 적용 필요
- Hard: 복합조건, 법률 교차 참조 필요

In [15]:
scenarios = []
sid = 0

def make_expected(so, school=False, repeat=False, extra=None):
    lv, info = get_severity_level(so)
    if info is None:
        return {'severity_level': None, 'criminal': False, 'legal_basis': '제156조'}
    exp = {'severity_level': lv, 'severity_label': info['label'],
           'criminal': info['criminal'], 'legal_basis': info['legal_basis'],
           'fine_amount': info.get('fine_van_etc'), 'demerit_points': info.get('demerit_points')}
    if school:
        exp.update({'aggravating':'school_zone','demerit_multiplier':2.0})
        dp = info.get('demerit_points')
        if dp is not None: exp['demerit_points_adjusted'] = dp * 2
    if repeat:
        exp.update({'aggravating':'repeat_100km','escalation':'제151조의2제2호','criminal':True})
    if extra: exp.update(extra)
    return exp

# ═══════════════════════════════════════
# A. Easy — 제한속도 명시, 단일 과속 (18건)
# ═══════════════════════════════════════
A_cases = [
    ('일반도로', 60, 75, 'general'),     # 15 → L1
    ('일반도로', 60, 95, 'general'),     # 35 → L2
    ('일반도로', 60, 110, 'general'),    # 50 → L3
    ('일반도로', 60, 130, 'general'),    # 70 → L4
    ('일반도로', 60, 150, 'general'),    # 90 → L5 (형사!)
    ('일반도로', 60, 170, 'general'),    # 110 → L6 (형사!)
    ('일반도로', 50, 65, 'general'),     # 15 → L1
    ('일반도로', 50, 80, 'general'),     # 30 → L2
    ('일반도로', 50, 105, 'general'),    # 55 → L3
    ('일반도로', 80, 145, 'general'),    # 65 → L4
    ('일반도로', 80, 165, 'general'),    # 85 → L5
    ('일반도로', 80, 195, 'general'),    # 115 → L6
    # ★ v4 추가 (6건): 다양한 속도·도로 조합
    ('일반도로', 50, 58, 'general'),     # 8 → L1
    ('일반도로', 80, 115, 'general'),    # 35 → L2
    ('일반도로', 50, 108, 'general'),    # 58 → L3
    ('일반도로', 50, 128, 'general'),    # 78 → L4
    ('일반도로', 80, 178, 'general'),    # 98 → L5
    ('일반도로', 50, 155, 'general'),    # 105 → L6
]
for road_kr, limit, actual, road in A_cases:
    sid += 1; so = actual - limit
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'A_easy_explicit_limit',
        'difficulty': 'easy', 'type': 'single', 'violation_types': ['speeding'],
        'description': f'화물차(4톤 초과)가 제한속도 {limit}km/h {road_kr}에서 {actual}km/h로 주행하였다.',
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': road, 'vehicle_type': 'cargo_over_4t', 'school_zone': False,
        'expected': make_expected(so)
    })

# ═══════════════════════════════════════
# B. Medium — 제한속도 미명시, 화물차 특수규정 필요 (18건)
# ═══════════════════════════════════════
B_cases = [
    ('고속도로', 80, 95, 'expressway'),    # 15→L1
    ('고속도로', 80, 115, 'expressway'),   # 35→L2
    ('고속도로', 80, 130, 'expressway'),   # 50→L3
    ('고속도로', 80, 150, 'expressway'),   # 70→L4
    ('고속도로', 80, 170, 'expressway'),   # 90→L5
    ('고속도로', 80, 190, 'expressway'),   # 110→L6
    ('자동차전용도로', 80, 100, 'auto_expressway'),  # 20→L1
    ('자동차전용도로', 80, 125, 'auto_expressway'),  # 45→L3
    ('일반도로', 60, 80, 'general'),       # 정확히 20→L1 (경계)
    ('일반도로', 60, 81, 'general'),       # 21→L2 (경계)
    ('일반도로', 60, 140, 'general'),      # 정확히 80→L4 (경계)
    ('일반도로', 60, 141, 'general'),      # 81→L5 (형사 전환 경계!)
    # ★ v4 추가 (6건): 경계값 추가 + 고속도로 변형
    ('일반도로', 60, 100, 'general'),      # 정확히 40→L2 (경계)
    ('일반도로', 60, 101, 'general'),      # 41→L3 (경계)
    ('일반도로', 60, 120, 'general'),      # 정확히 60→L3 (경계)
    ('일반도로', 60, 121, 'general'),      # 61→L4 (경계)
    ('일반도로', 60, 160, 'general'),      # 정확히 100→L5 (경계)
    ('일반도로', 60, 161, 'general'),      # 101→L6 (형사 전환 경계!)
]
for road_kr, limit, actual, road in B_cases:
    sid += 1; so = actual - limit
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'B_medium_no_limit',
        'difficulty': 'medium', 'type': 'single', 'violation_types': ['speeding'],
        'description': f'화물차(4톤 초과)가 {road_kr}에서 {actual}km/h로 주행하였다.',
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': road, 'vehicle_type': 'cargo_over_4t', 'school_zone': False,
        'expected': make_expected(so)
    })

# ═══════════════════════════════════════
# C. Easy — 비과속 위반 (14건)
# ═══════════════════════════════════════
# ★ 설계 의도: 비과속 위반(급가속, 급감속 등)은 과속 6단계 심각도(SGT)에 해당하지 않음.
# expected severity_level=None 으로 설정. M3 Severity F1 계산 시 '_normalize_severity()'
# 함수에서 None → 'none' 문자열로 정규화되어 별도 클래스로 처리됨.
# 법적 근거: 도로교통법 제156조 (20만원 이하 벌금/구류/과료) — 범칙금 차등 없음.
C_cases = [
    ('sudden_accel','급가속','화물차가 일반도로에서 급가속하였다 (초당 가속도 +9km/h).'),
    ('sudden_start','급출발','화물차가 정지 상태에서 급출발하였다 (초당 가속도 +11km/h).'),
    ('sudden_decel','급감속','화물차가 50km/h 주행 중 급감속하였다 (초당 감속 -15km/h).'),
    ('sudden_stop','급정지','화물차가 40km/h 주행 중 급정지하였다 (감속 후 5km/h 이하).'),
    ('sudden_lane_change','급진로변경','화물차가 35km/h 주행 중 급격히 차로를 변경하였다.'),
    ('sudden_overtake','급앞지르기','화물차가 급차로변경과 동시에 가속하며 앞지르기하였다.'),
    ('sudden_turn','급회전','화물차가 25km/h로 교차로에서 급회전하였다.'),
    ('sudden_uturn','급유턴','화물차가 20km/h로 급유턴하였다.'),
    # ★ v4 추가 (6건): 다른 속도/상황
    ('sudden_accel','급가속','화물차가 고속도로 진입로에서 급가속하였다 (초당 가속도 +12km/h).'),
    ('sudden_decel','급감속','화물차가 80km/h 고속도로 주행 중 급감속하였다 (초당 감속 -18km/h).'),
    ('sudden_stop','급정지','화물차가 60km/h 주행 중 갑작스럽게 급정지하였다.'),
    ('sudden_lane_change','급진로변경','화물차가 70km/h 고속도로에서 급격히 차로를 변경하였다.'),
    ('sudden_turn','급좌회전','화물차가 22km/h로 교차로에서 급좌회전하였다.'),
    ('sudden_turn','급우회전','화물차가 18km/h로 교차로에서 급우회전하였다.'),
]
for vtype, kr, desc in C_cases:
    sid += 1
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'C_easy_non_speed',
        'difficulty': 'easy', 'type': 'single', 'violation_types': [vtype],
        'description': desc, 'speed_over': None, 'road_type': 'general',
        'vehicle_type': 'cargo_over_4t', 'school_zone': False,
        'expected': {'severity_level': None, 'criminal': False, 'legal_basis': '제156조'}
    })

# ═══════════════════════════════════════
# D. Hard — 복합 위반 (14건)
# ═══════════════════════════════════════
D_cases = [
    (['speeding','sudden_decel'], 60, 130, 'general',
     '화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 고속 주행 후 급감속하였다. 주행속도 130km/h.'),
    (['speeding','sudden_stop'], 60, 110, 'general',
     '화물차(4톤 초과)가 제한속도 60km/h 도로에서 110km/h 주행 중 급정지하였다.'),
    (['speeding','sudden_lane_change'], 80, 150, 'expressway',
     '화물차(4톤 초과)가 고속도로에서 150km/h로 주행하며 급진로변경하였다.'),
    (['speeding','sudden_accel'], 60, 95, 'general',
     '화물차(4톤 초과)가 일반도로에서 급가속하여 95km/h에 도달하였다. 해당 도로 제한속도 60km/h.'),
    (['speeding','sudden_overtake'], 80, 125, 'expressway',
     '화물차(4톤 초과)가 고속도로에서 125km/h로 급앞지르기하였다.'),
    (['sudden_accel','sudden_decel'], None, None, 'general',
     '화물차가 일반도로에서 급가속 직후 급감속하였다 (지그재그 운전).'),
    (['sudden_lane_change','sudden_decel'], None, None, 'general',
     '화물차가 일반도로에서 급진로변경 후 급감속하였다.'),
    (['speeding','sudden_turn'], 50, 85, 'general',
     '화물차(4톤 초과)가 제한속도 50km/h 교차로에서 85km/h로 급회전하였다.'),
    (['speeding','sudden_decel','sudden_stop'], 60, 140, 'general',
     '화물차(4톤 초과)가 일반도로에서 140km/h 주행 후 급감속하여 급정지하였다. 제한속도 60km/h.'),
    (['speeding','sudden_lane_change','sudden_overtake'], 80, 135, 'expressway',
     '화물차(4톤 초과)가 고속도로에서 135km/h로 주행하며 급진로변경과 급앞지르기를 동시에 하였다.'),
    # ★ v4 추가 (4건)
    (['speeding','sudden_uturn'], 50, 72, 'general',
     '화물차(4톤 초과)가 제한속도 50km/h 도로에서 72km/h로 주행 중 급유턴을 시도하였다.'),
    (['speeding','sudden_accel','sudden_lane_change'], 60, 105, 'general',
     '화물차(4톤 초과)가 일반도로에서 급가속하여 105km/h에 도달 후 급진로변경하였다. 제한속도 60km/h.'),
    (['sudden_accel','sudden_turn'], None, None, 'general',
     '화물차가 교차로 진입 시 급가속 후 급회전하였다.'),
    (['sudden_lane_change','sudden_overtake'], None, None, 'expressway',
     '화물차가 고속도로에서 급진로변경과 동시에 급앞지르기를 하였다.'),
]
for vtypes, limit, actual, road, desc in D_cases:
    sid += 1; so = (actual-limit) if limit and actual else None
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'D_hard_compound',
        'difficulty': 'hard', 'type': 'compound', 'violation_types': vtypes,
        'description': desc,
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': road, 'vehicle_type': 'cargo_over_4t', 'school_zone': False,
        'expected': make_expected(so, extra={'compound_violations':len(vtypes)}) if so else
                   {'severity_level':None,'criminal':False,'legal_basis':'제156조','compound_violations':len(vtypes)}
    })

# ═══════════════════════════════════════
# E. Hard — 가중 조건 (14건)
# ═══════════════════════════════════════
E_cases = [
    (30, 55, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 55km/h로 주행하였다.'),
    (30, 75, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 75km/h로 주행하였다.'),
    (30, 100, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 100km/h로 주행하였다.'),
    (30, 140, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 140km/h로 주행하였다.'),
    (60, 170, False, True,
     '화물차(4톤 초과)가 일반도로에서 170km/h로 주행하였다. 해당 운전자는 이전에 100km/h 초과 과속으로 2회 적발된 바 있다.'),
    (80, 190, False, True,
     '화물차(4톤 초과)가 고속도로에서 190km/h로 주행하였다. 해당 운전자는 과거 100km/h 초과 과속 2회 전력이 있다.'),
    (60, 95, False, False,
     '적재중량 1.5톤 초과 화물차가 일반도로에서 95km/h로 주행하였다. 해당 도로 제한속도 60km/h.'),
    (60, 130, False, False,
     '적재중량 1.5톤 이하 소형화물차가 일반도로에서 130km/h로 주행하였다. 해당 도로 제한속도 60km/h.'),
    # ★ v4 추가 (6건): 보호구역 레벨 보강 + 반복과속
    (30, 45, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 45km/h로 주행하였다.'),
    (30, 120, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 120km/h로 주행하였다.'),
    (30, 50, True, False,
     '화물차(4톤 초과)가 스쿨존(제한속도 30km/h)에서 50km/h로 주행하였다.'),
    (30, 160, True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 160km/h로 주행하였다.'),
    (60, 165, False, True,
     '화물차(4톤 초과)가 일반도로에서 165km/h로 주행하였다. 제한속도 60km/h. 해당 운전자는 100km/h 초과 과속으로 3회 적발 전력이 있다.'),
    (80, 185, False, True,
     '화물차(4톤 초과)가 고속도로에서 185km/h로 주행하였다. 해당 운전자는 과거 100km/h 초과 과속 전력이 3회이다.'),
]
for limit, actual, school, repeat, desc in E_cases:
    sid += 1; so = actual - limit
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'E_hard_aggravating',
        'difficulty': 'hard', 'type': 'aggravated', 'violation_types': ['speeding'],
        'description': desc,
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': 'school_zone' if school else ('expressway' if limit>=80 else 'general'),
        'vehicle_type': 'cargo_over_4t', 'school_zone': school, 'repeat_over_100': repeat,
        'expected': make_expected(so, school=school, repeat=repeat)
    })

# ═══════════════════════════════════════
# F. Medium — 특수 도로/조건 (10건)
# ═══════════════════════════════════════
# ★ 법적 근거 주석:
# - 기상악화(비·안개·눈) 시 제한속도 20% 감속: 도로교통법 시행규칙 제19조 제1항 제2호
#   '비가 내려 노면이 젖어 있는 경우' → 최고속도의 100분의 20을 줄인 속도
# - 화물차 고속도로 제한속도 80km/h × 0.8 = 64km/h (폭우 시나리오 S067)
F_cases = [
    (80, 130, 'auto_only', ['speeding'],
     '화물차(4톤 초과)가 자동차전용도로에서 130km/h로 주행하였다.'),
    (60, 90, 'general', ['speeding'],
     '화물차(4톤 초과, DTG 미제출 상태)가 일반도로에서 90km/h로 주행하였다. 제한속도 60km/h.'),
    (64, 109, 'expressway', ['speeding'],
     '화물차(4톤 초과)가 폭우 중 고속도로에서 109km/h로 주행하였다. 기상악화 시 제한속도 20% 감속 적용(시행규칙 제19조 제1항 제2호). 화물차 고속도로 80km/h × 0.8 = 64km/h.'),
    (50, 95, 'construction', ['speeding'],
     '화물차(4톤 초과)가 공사구간(제한속도 50km/h)에서 95km/h로 주행하였다.'),
    (70, 140, 'tunnel', ['speeding','sudden_decel'],
     '화물차(4톤 초과)가 터널(제한속도 70km/h)에서 140km/h로 주행 후 급감속하였다.'),
    # ★ v4 추가 (5건)
    (40, 75, 'residential', ['speeding'],
     '화물차(4톤 초과)가 주택가(제한속도 40km/h)에서 75km/h로 주행하였다.'),
    (50, 110, 'construction', ['speeding','sudden_stop'],
     '화물차(4톤 초과)가 공사구간(제한속도 50km/h)에서 110km/h 주행 후 급정지하였다.'),
    (60, 85, 'general', ['speeding'],
     '화물차(4톤 초과, 야간 운행)가 일반도로에서 85km/h로 주행하였다. 제한속도 60km/h.'),
    (80, 145, 'tunnel', ['speeding'],
     '화물차(4톤 초과)가 터널구간(제한속도 80km/h)에서 145km/h로 주행하였다.'),
    (60, 75, 'general', ['speeding'],
     '화물차(4톤 초과)가 비 오는 날 일반도로(제한속도 60km/h)에서 75km/h로 주행하였다.'),
]
for limit, actual, road, vtypes, desc in F_cases:
    sid += 1; so = actual-limit
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'F_medium_special',
        'difficulty': 'medium', 'type': 'compound' if len(vtypes)>1 else 'single',
        'violation_types': vtypes, 'description': desc,
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': road, 'vehicle_type': 'cargo_over_4t', 'school_zone': False,
        'expected': make_expected(so)
    })

# ═══════════════════════════════════════
# G. Hard — 법률 교차 (12건)
# ═══════════════════════════════════════
# ★★ G 카테고리: 교차 법령 Multi-hop 필요 시나리오
# S090: 과속 + DTG 미제출 → 도로교통법 + 교통안전법 제55조 (교차 법령 Multi-hop)
# S091: 과속 + 교육 미이수 → 도로교통법 + 화물자동차운수사업법 제59조 (교차 법령 Multi-hop)
# S095: 과속 + 적재량 초과 → 도로교통법 제17조 + 제49조 (동일 법령 내 교차 조문)
# 이 설계로 M6 Multi-hop Recall이 S3/S4의 KG 교차 법령 추론 능력을 직접 측정
G_cases = [
    (60, 85, 25, ['speeding'], False, False,
     '화물차(4톤 초과)가 일반도로(제한속도 60km/h)에서 85km/h 주행 중 교통사고를 유발하였다.'),
    # ★ dtg_violation 추가 — 교통안전법 제55조 교차 참조 (KG Multi-hop 필요)
    (60, 110, 50, ['speeding', 'dtg_violation'], False, False,
     '화물차(4톤 초과, DTG 운행기록 미제출)가 일반도로에서 110km/h로 주행하였다. 제한속도 60km/h.'),
    # ★ education 추가 — 화물자동차운수사업법 제59조 교차 참조 (KG Multi-hop 필요)
    (60, 130, 70, ['speeding', 'education'], False, False,
     '화물차(4톤 초과, 운수종사자 교육 미이수)가 일반도로에서 130km/h로 주행하였다. 제한속도 60km/h.'),
    (30, 70, 40, ['speeding', 'sudden_stop'], True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 70km/h로 주행 중 급정지하였다.'),
    (60, 150, 90, ['speeding'], False, True,
     '화물차(4톤 초과)가 일반도로에서 150km/h로 주행하였다. 제한속도 60km/h. 이 운전자는 100km/h 초과 과속 전력 2회.'),
    # ★ v4 추가 (7건)
    (30, 95, 65, ['speeding', 'sudden_decel'], True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 95km/h로 주행 중 급감속하였다.'),
    # ★ overload 추가 — 도로교통법 제49조 교차 조문 (동일 법령 내 Multi-hop)
    (60, 100, 40, ['speeding', 'overload'], False, False,
     '화물차(4톤 초과, 적재량 초과 상태)가 일반도로에서 100km/h로 주행하였다. 제한속도 60km/h.'),
    (80, 165, 85, ['speeding'], False, False,
     '화물차(4톤 초과)가 고속도로에서 165km/h로 주행 중 교통사고를 유발하였다.'),
    (50, 120, 70, ['speeding', 'sudden_lane_change'], False, False,
     '화물차(4톤 초과)가 일반도로(제한속도 50km/h)에서 120km/h로 주행하며 급진로변경하였다.'),
    (60, 175, 115, ['speeding'], False, True,
     '화물차(4톤 초과)가 일반도로에서 175km/h로 주행하였다. 제한속도 60km/h. 운전자는 100km/h 초과 과속 3회 적발 전력.'),
    (30, 85, 55, ['speeding', 'sudden_accel'], True, False,
     '화물차(4톤 초과)가 어린이보호구역에서 급가속하여 85km/h에 도달하였다.'),
    (80, 145, 65, ['speeding', 'sudden_stop'], False, False,
     '화물차(4톤 초과)가 고속도로에서 145km/h 주행 중 급정지하였다.'),
]
for limit, actual, so, vtypes, school, repeat, desc in G_cases:
    sid += 1
    scenarios.append({
        'id': f'S{sid:03d}', 'category': 'G_hard_cross_law',
        'difficulty': 'hard', 'type': 'compound', 'violation_types': vtypes,
        'description': desc,
        'speed_limit': limit, 'actual_speed': actual, 'speed_over': so,
        'road_type': 'school_zone' if school else ('expressway' if limit>=80 else 'general'),
        'vehicle_type': 'cargo_over_4t', 'school_zone': school,
        'expected': make_expected(so, school=school, repeat=repeat)
    })

# ── 저장 ──
save_json(scenarios, SCENARIO_DIR / 'scenarios_100.json')
# 하위 호환: 60건 버전도 저장
save_json(scenarios[:60], SCENARIO_DIR / 'scenarios_60.json')

print(f'\n=== 시나리오: {len(scenarios)}건 ===')
for cat, cnt in sorted(Counter(s['category'] for s in scenarios).items()):
    print(f'  {cat}: {cnt}건')
diff_counts = Counter(s['difficulty'] for s in scenarios)
print(f'\n난이도: Easy {diff_counts["easy"]}건, Medium {diff_counts["medium"]}건, Hard {diff_counts["hard"]}건')

# 레벨 분포 확인
from collections import Counter
lv_dist = Counter(s['expected'].get('severity_level') for s in scenarios)
print(f'\n심각도 레벨 분포:')
for lv in sorted(lv_dist.keys(), key=lambda x: str(x)):
    print(f'  {lv}: {lv_dist[lv]}건')

# 변별력 포인트 확인
medium_hw = [s for s in scenarios if s['category']=='B_medium_no_limit' and s['road_type']=='expressway']
print(f'\n[변별력] 고속도로 화물차 시나리오 (S1 오답 예상): {len(medium_hw)}건')
for s in medium_hw[:3]:
    print(f'  {s["id"]}: {s["description"][:60]}... → 정답: {s["expected"]["severity_level"]}')

boundary = [s for s in scenarios if s['category']=='B_medium_no_limit'
            and s.get('speed_over') in [20,21,40,41,60,61,80,81,100,101]]
print(f'\n[변별력] 경계값 테스트: {len(boundary)}건')

print(f'\n→ 다음: 03_legal_kg_construction.ipynb')


✅ 저장: data\scenarios\scenarios_100.json
✅ 저장: data\scenarios\scenarios_60.json

=== 시나리오: 100건 ===
  A_easy_explicit_limit: 18건
  B_medium_no_limit: 18건
  C_easy_non_speed: 14건
  D_hard_compound: 14건
  E_hard_aggravating: 14건
  F_medium_special: 10건
  G_hard_cross_law: 12건

난이도: Easy 32건, Medium 28건, Hard 40건

심각도 레벨 분포:
  None: 18건
  level_1: 9건
  level_2: 17건
  level_3: 18건
  level_4: 17건
  level_5: 9건
  level_6: 12건

[변별력] 고속도로 화물차 시나리오 (S1 오답 예상): 6건
  S019: 화물차(4톤 초과)가 고속도로에서 95km/h로 주행하였다.... → 정답: level_1
  S020: 화물차(4톤 초과)가 고속도로에서 115km/h로 주행하였다.... → 정답: level_2
  S021: 화물차(4톤 초과)가 고속도로에서 130km/h로 주행하였다.... → 정답: level_3

[변별력] 경계값 테스트: 11건

→ 다음: 03_legal_kg_construction.ipynb


In [16]:
# ★★ 시나리오 생성 품질 검증
# 1) speed_over 필드 정확성 (04에서 Python 계산으로 사용)
# 2) get_severity_level 경계값 테스트 (법령: 이하/초과 = inclusive_max/exclusive_min)
# 3) G 카테고리 교차 법령 violation_types 확인

from collections import Counter

print('=== 1. speed_over 정확성 검증 ===')
mismatch = []
for sc in scenarios:
    if sc.get('actual_speed') and sc.get('speed_limit'):
        expected_so = sc['actual_speed'] - sc['speed_limit']
        stored_so = sc.get('speed_over')
        if stored_so != expected_so:
            mismatch.append(f"{sc['id']}: stored={stored_so}, calc={expected_so}")
if mismatch:
    print(f'  ❌ 불일치 {len(mismatch)}건: {mismatch[:3]}')
else:
    speed_sc = sum(1 for s in scenarios if s.get('actual_speed'))
    print(f'  ✅ {speed_sc}건 speed_over 정확 — 04에서 Python 계산으로 직접 사용 가능')

print('\n=== 2. get_severity_level 경계값 단위 테스트 ===')
test_cases = [
    (20, 'level_1', '20km/h 이하=L1'),
    (21, 'level_2', '21km/h → L2 시작'),
    (40, 'level_2', '40km/h 이하=L2'),
    (41, 'level_3', '41km/h → L3 시작'),
    (60, 'level_3', '60km/h 이하=L3'),
    (61, 'level_4', '61km/h → L4 시작'),
    (80, 'level_4', '80km/h 이하=L4'),
    (81, 'level_5', '81km/h → L5 형사처벌'),
    (100, 'level_5', '100km/h 이하=L5'),
    (101, 'level_6', '101km/h → L6 가중처벌'),
]
all_pass = True
for so, expected, desc in test_cases:
    lv, info = get_severity_level(so)
    ok = lv == expected
    if not ok: all_pass = False
    print(f'  {"✅" if ok else "❌ FAIL"} so={so:3d} → {lv} ({desc})')
if all_pass:
    print('  → 경계값 10건 전부 통과 ✅')
    print('  → Neo4j 쿼리: WHERE s.speed_over_min < $so AND s.speed_over_max >= $so')
else:
    print('  ❌ 실패! config.py get_severity_level() 수정 필요')
    print('  → if so <= 20: L1 / elif so <= 40: L2 ... (<=, inclusive_max)')

print('\n=== 3. G 카테고리 교차 법령 확인 ===')
cross_law = [(sc['id'], sc['violation_types']) for sc in scenarios
             if sc['category'] == 'G_hard_cross_law' and len(sc['violation_types']) > 1]
for sid, vtypes in cross_law:
    print(f'  {sid}: {vtypes}')
print(f'  → G 카테고리 교차 법령 시나리오: {len(cross_law)}건')
print(f'  → S4가 TSA_55, TTBA_59, RTA_49를 인용 시 M6 Recall 향상')

print('\n=== 4. 전체 시나리오 분포 ===')
cat_cnt = Counter(s['category'] for s in scenarios)
diff_cnt = Counter(s['difficulty'] for s in scenarios)
lv_cnt = Counter(s['expected'].get('severity_level') for s in scenarios)
for cat, cnt in sorted(cat_cnt.items()):
    print(f'  {cat}: {cnt}건')
print(f'  난이도: Easy={diff_cnt["easy"]}, Medium={diff_cnt["medium"]}, Hard={diff_cnt["hard"]}')
print(f'  심각도: ' + ', '.join(f'{k}={v}' for k,v in sorted(lv_cnt.items(), key=lambda x: str(x[0]))))


=== 1. speed_over 정확성 검증 ===
  ✅ 82건 speed_over 정확 — 04에서 Python 계산으로 직접 사용 가능

=== 2. get_severity_level 경계값 단위 테스트 ===
  ✅ so= 20 → level_1 (20km/h 이하=L1)
  ✅ so= 21 → level_2 (21km/h → L2 시작)
  ✅ so= 40 → level_2 (40km/h 이하=L2)
  ✅ so= 41 → level_3 (41km/h → L3 시작)
  ✅ so= 60 → level_3 (60km/h 이하=L3)
  ✅ so= 61 → level_4 (61km/h → L4 시작)
  ✅ so= 80 → level_4 (80km/h 이하=L4)
  ✅ so= 81 → level_5 (81km/h → L5 형사처벌)
  ✅ so=100 → level_5 (100km/h 이하=L5)
  ✅ so=101 → level_6 (101km/h → L6 가중처벌)
  → 경계값 10건 전부 통과 ✅
  → Neo4j 쿼리: WHERE s.speed_over_min < $so AND s.speed_over_max >= $so

=== 3. G 카테고리 교차 법령 확인 ===
  S090: ['speeding', 'dtg_violation']
  S091: ['speeding', 'education']
  S092: ['speeding', 'sudden_stop']
  S094: ['speeding', 'sudden_decel']
  S095: ['speeding', 'overload']
  S097: ['speeding', 'sudden_lane_change']
  S099: ['speeding', 'sudden_accel']
  S100: ['speeding', 'sudden_stop']
  → G 카테고리 교차 법령 시나리오: 8건
  → S4가 TSA_55, TTBA_59, RTA_49를 인용 시 M6 Recall 향상

=== 4. 전체 시나